# PART 2: PRO vs PRE Schema Comparison
Production Environment Schema Registry Validator

This notebook compares table schemas between PRO and PRE
environments to identify synchronization gaps.

**Prerequisites:**
- Part 1 completed (DDL registry from PRO exists)
- Executed in PRE workspace


In [ ]:

# ======================================================
# 1. Imports and Configuration
# ======================================================
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from datetime import datetime
import json
import re
from collections import Counter


CONFIG = {
    # Source table from Part 1
    "ddl_registry_pro": "metadata.ddl_registry_pro",

    # Output table for this analysis
    "comparison_results_table": "metadata.schema_comparison_pro_pre",

    # Environment identifier
    "current_environment": "PRE",

    # Analysis metadata
    "analysis_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
    "analysis_date": datetime.now().strftime("%Y-%m-%d"),

    # Exclusion patterns
    "exclude_table_patterns": [
        r".*bkp.*", r".*backup.*", r".*temp.*", r".*tmp.*",
        r".*clone.*", r".*test.*", r".*old.*", r".*dev.*"
    ],
    "exclude_database_patterns": [
        r".*backup.*", r".*bkp.*", r".*temp.*", r".*tmp.*",
        r".*test.*", r".*old.*", r".*dev.*"
    ]
}
# Display configuration summary
print("=" * 80)
print("CONFIGURATION - PART 2: SCHEMA COMPARISON")
print("=" * 80)
print(f"Current Environment: {CONFIG['current_environment']}")
print(f"PRO DDL Source: {CONFIG['ddl_registry_pro']}")
print(f"Results Destination: {CONFIG['comparison_results_table']}")
print(f"Analysis ID: {CONFIG['analysis_id']}")
print("=" * 80)



In [ ]:

# ======================================================
# 2. Metadata Keywords Registry
# ======================================================
# These keywords identify metadata rows from DESCRIBE
# outputs and must NOT be treated as real columns.
# ======================================================

METADATA_KEYWORDS = {
    "catalog", "database", "table", "owner", "created time",
    "last access", "created by", "type", "provider", "location",
    "comment", "serde library", "inputformat", "outputformat",
    "storage properties", "table parameters", "serde parameters",
    "partition provider", "partition columns", "not partitioned",
    "# partition information", "column names",
    "table properties", "statistics", "view text",
    "view original text", "view expanded text",
    "# detailed table information",
    "delta.minreaderversion", "delta.minwriterversion",
    "transient_lastddltime"
}


In [ ]:


# ======================================================
# 3. Metadata Detection Utilities
# ======================================================



def is_metadata_field(col_name, data_type=""):
    """
    Determines whether a DESCRIBE row represents metadata
    rather than a real data column.
    """
    if not col_name:
        return True

    col_lower = col_name.lower().strip()

    if not col_lower or col_lower.startswith("#"):
        return True

    if col_lower in {"col_name", "data_type", "comment"}:
        return True

    if col_lower in METADATA_KEYWORDS:
        return True

    if re.match(r"^part\s*\d+$", col_lower):
        return True

    if "=" in col_name or "->" in col_name:
        return True

    return False


def extract_partition_columns_from_ddl(ddl):
    """
    Extracts partition column names from a CREATE TABLE DDL.
    """
    if not ddl:
        return []

    match = re.search(r"PARTITIONED BY\s*\(([^)]+)\)", ddl, re.IGNORECASE)
    if not match:
        return []

    parts = match.group(1).split(",")
    return [p.strip().split()[0].lower() for p in parts]



In [ ]:

# ======================================================
# 4. PRE Schema Extraction
# ======================================================


def get_table_schema_pre(database, table):
    """
    Retrieves actual data columns from PRE.
    Views are ignored.
    """
    full_name = f"{database}.{table}"

    try:
        desc_ext = spark.sql(f"DESCRIBE EXTENDED {full_name}").collect()
        for row in desc_ext:
            if row.col_name == "Type" and "VIEW" in row.data_type.upper():
                return None

        desc = spark.sql(f"DESCRIBE {full_name}").collect()

        columns = []
        partition_cols = []
        in_partition = False

        for row in desc:
            name = row.col_name.strip() if row.col_name else ""
            dtype = row.data_type.strip() if row.data_type else ""

            if name == "# Partition Information":
                in_partition = True
                continue

            if is_metadata_field(name, dtype):
                continue

            if in_partition:
                partition_cols.append({"name": name, "type": dtype})
            else:
                columns.append({"name": name, "type": dtype})

        return {
            "columns": columns,
            "partition_columns": partition_cols
        }

    except Exception as e:
        print(f"ERROR reading {full_name}: {e}")
        return None


In [ ]:

# ======================================================
# 5. Column Cleaning
# ======================================================

def clean_columns(columns, partition_names):
    """
    Removes metadata and partition columns.
    """
    partition_names = [p.lower() for p in partition_names]

    cleaned = []
    for col in columns:
        name = col["name"].strip()
        dtype = col["type"].strip()

        if is_metadata_field(name, dtype):
            continue

        if name.lower() in partition_names:
            continue

        cleaned.append(col)

    return cleaned



In [ ]:

# ======================================================
# 6. Schema Comparison Engine
# ======================================================


def compare_schemas(cols_pro_json, database, table, ddl_pro):
    """
    Compares PRO and PRE schemas column by column.
    """
    result = {
        "schemas_match": False,
        "columns_only_in_pro": [],
        "columns_only_in_pre": [],
        "columns_with_different_types": [],
        "error": None
    }

    try:
        partition_cols = extract_partition_columns_from_ddl(ddl_pro)
        cols_pro = clean_columns(json.loads(cols_pro_json), partition_cols)

        pre_schema = get_table_schema_pre(database, table)
        if pre_schema is None:
            result["error"] = "VIEW or inaccessible table"
            return result

        cols_pre = pre_schema["columns"]

        pro = {c["name"].lower(): c["type"].lower() for c in cols_pro}
        pre = {c["name"].lower(): c["type"].lower() for c in cols_pre}

        for k in pro:
            if k not in pre:
                result["columns_only_in_pro"].append({"name": k, "type": pro[k]})
            elif pro[k] != pre[k]:
                result["columns_with_different_types"].append({
                    "name": k,
                    "type_pro": pro[k],
                    "type_pre": pre[k]
                })

        for k in pre:
            if k not in pro:
                result["columns_only_in_pre"].append({"name": k, "type": pre[k]})

        result["schemas_match"] = (
            not result["columns_only_in_pro"] and
            not result["columns_only_in_pre"] and
            not result["columns_with_different_types"]
        )

    except Exception as e:
        result["error"] = str(e)

    return result



In [ ]:

# ======================================================
# 7. Filtering Utilities
# ======================================================


def should_exclude_database(db):
    return any(re.match(p, db.lower()) for p in CONFIG["exclude_database_patterns"])


def should_exclude_table(table):
    return any(re.match(p, table.lower()) for p in CONFIG["exclude_table_patterns"])



In [ ]:

# ======================================================
# 9. Execute Analysis
# ======================================================

analysis_results = []

for r in tables_to_analyze:
    result = {
        "analysis_id": CONFIG["analysis_id"],
        "analysis_date": CONFIG["analysis_date"],
        "database_pro": r.database,
        "database_pre": r.database,
        "table_name": r.table_name,
        "full_table_name_pro": r.full_table_name,
        "full_table_name_pre": f"{r.database}.{r.table_name}",
        "database_exists_pre": False,
        "table_exists_pre": False,
        "schemas_match": False,
        "requires_action": True,
        "action_type": None,
        "schema_differences": None,
        "error_message": None
    }

    try:
        dbs = [d.databaseName for d in spark.sql("SHOW DATABASES").collect()]
        if r.database not in dbs:
            result["action_type"] = "CREATE_DATABASE_AND_TABLE"
            analysis_results.append(result)
            continue

        result["database_exists_pre"] = True

        tables = [t.tableName for t in spark.sql(f"SHOW TABLES IN {r.database}").collect()]
        if r.table_name not in tables:
            result["action_type"] = "CREATE_TABLE"
            analysis_results.append(result)
            continue

        result["table_exists_pre"] = True

        cmp = compare_schemas(r.columns, r.database, r.table_name, r.ddl)

        if cmp["error"]:
            result["action_type"] = "ERROR"
            result["error_message"] = cmp["error"]
        elif cmp["schemas_match"]:
            result["action_type"] = "NO_ACTION"
            result["schemas_match"] = True
            result["requires_action"] = False
        else:
            result["action_type"] = "UPDATE_SCHEMA"
            result["schema_differences"] = json.dumps(cmp)

    except Exception as e:
        result["action_type"] = "ERROR"
        result["error_message"] = str(e)

    analysis_results.append(result)



In [ ]:

# ======================================================
# 10. Save Results
# ======================================================

schema = StructType([
    StructField("analysis_id", StringType(), False),
    StructField("analysis_date", StringType(), False),
    StructField("database_pro", StringType(), False),
    StructField("database_pre", StringType(), False),
    StructField("table_name", StringType(), False),
    StructField("full_table_name_pro", StringType(), False),
    StructField("full_table_name_pre", StringType(), False),
    StructField("database_exists_pre", BooleanType(), False),
    StructField("table_exists_pre", BooleanType(), False),
    StructField("schemas_match", BooleanType(), False),
    StructField("requires_action", BooleanType(), False),
    StructField("action_type", StringType(), True),
    StructField("schema_differences", StringType(), True),
    StructField("error_message", StringType(), True),
])

df = spark.createDataFrame(analysis_results, schema)

spark.sql("CREATE DATABASE IF NOT EXISTS metadata")

df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(CONFIG["comparison_results_table"])

print("PART 2 COMPLETED SUCCESSFULLY")
